# Hands-on Jev Tutorial: From Concepts to Building AI Workflows

You learn Jev by building with it, not by reading about it.

You'll use TypeSafe AI's **Jev** model, served from your own Databricks workspace through the Unity AI Gateway. The path runs from the bare idea to a simple decision system you could use as reference to build extended decision based system with Jev.

I assumes you want both halves of the story: how Jev works, and why you would reach for it instead of a conventional LLM. We will explore with the architecture concepts, brings in code once code earns its place, and close every lesson with an short exercise.

The tutorial follows six lessons:

| Lesson | What you learn | Code? |
|---|---|---|
| 1 | Why TypeSafe built Jev, and the judgment/action split | no |
| 2 | The three primitives: `Noul`, `Choice`, `Score` | reading |
| 3 | Your first API call | yes |
| 4 | Asking several questions in one call | yes |
| 5 | Working with uncertainty: confidence and thresholds | yes |
| 6 | Building a sample AI decision-based workflow | yes |

Each lesson ends with an exercise and a link into the [official Jev documentation](https://docs.typesafe.ai/).

Let's start with setting up the enviorment. I assume that you have access to a Databricks Workspace and Unity AI Gateway.

## Setup for Databricks Workspace and Unity AI Gateway Endpoints

Jev here is served inside your own Databricks workspace through the **Unity AI Gateway**, so there's no SDK to install and no second vendor account to create. Nor do you need TypeSafe API keys. The credentials this repo already uses are the only ones you need.

Put these four in `jevai/.env`:

```
DATABRICKS_HOST=https://your-workspace.cloud.databricks.com
DATABRICKS_TOKEN=dapi...
JEV_MODEL=catalog.schema.your_jev_endpoint
JEVAI_MODEL=catalog.schema.your_chat_endpoint
```

`JEV_MODEL` is the **Unity Catalog endpoint path** for the registered Jev model. Jev has its own gateway route.

![One host, two routes: in the notebook jev_client.py POSTs state and typed questions to the Unity AI Gateway route /ai-gateway/typesafe/v1/systemone, which reaches the Jev System One Unity Catalog endpoint named by JEV_MODEL and returns typed answers that router.decide() branches on; separately, Lesson 6.2 POSTs to the same gateway's /ai-gateway/mlflow/v1 route with the same token to reach a Unity Catalog chat endpoint that writes prose only. DATABRICKS_HOST and DATABRICKS_TOKEN serve both routes, JEV_MODEL is not AI_GATEWAY_BASE_URL, and the openai package in the second route is only the client library rather than a call to OpenAI](images/jev_gateway_path.png)

`JEVAI_MODEL` is the chat endpoint Lesson 6.2 uses for the customer-facing reply, reached through the same gateway on its `/ai-gateway/mlflow/v1` route. Like `JEV_MODEL` it is a Unity Catalog path rather than a model name, and the two are different endpoints on different routes. Only Lesson 6 reads it, so Lessons 1 through 5 run without it. `JEVAI_GATEWAY_URL` overrides the route if yours differs.

`JevClient()` reads the first three with no arguments, and it loads `jevai/.env` itself. Your credentials hold even if you run a cell out of order, or leave a kernel over from an earlier session.

**If you change `jevai/.env` or any file under `jevai/`, restart the kernel.** Jupyter caches imported modules, so an edit to `jev_client.py` is invisible to a running kernel until then. Lessons 1 and 2 are reading only, so you can start before any of this is in place. The cell below is the first thing that needs the credentials, so skip it now and run it when you reach Lesson 3.

In [18]:
import sys
from pathlib import Path

# Make `import jevai...` work whether you started Jupyter in the repo root or
# inside jevai/. This is the only wiring the notebook needs.
for d in [Path.cwd(), *Path.cwd().parents]:
    if (d / 'jevai' / 'jev_client.py').exists():
        sys.path.insert(0, str(d))
        break
else:
    raise FileNotFoundError(f'Could not find the jevai package from {Path.cwd()}')

# jev_client.py loads jevai/.env itself, from next to its own file. Nothing to
# set up here, and a stale kernel can't leave the client on someone else's token.
from jevai.jev_client import ENV_FILE, Choice, JevClient, Noul, Score

client = JevClient()

# Printed so a wrong file or endpoint is obvious here, not three cells later.
print('env file :', ENV_FILE)
print('endpoint :', client.model)
print('gateway  :', client.url)

# The customer message this tutorial carries all the way through.
TICKET = (
    'I canceled my subscription last week, but you charged my credit card '
    'again today. Please fix this immediately. This is not acceptable.'
)
print()
print(TICKET)


env file : /Users/jules/git-repos/mlflow-genai-tutorials/jevai/.env
endpoint : jules_catalog.uaigw_jevl.uaigw_jev1_endpoint
gateway  : https://e2-dogfood.staging.cloud.databricks.com/ai-gateway/typesafe/v1/systemone

I canceled my subscription last week, but you charged my credit card again today. Please fix this immediately. This is not acceptable.


---

# Lesson 1: Why did TypeSafe build Jev?

## 1.1 Start with a real problem

Imagine you're building an AI customer-service agent for an online retailer.

A customer writes:

> I canceled my subscription last week, but you charged my credit card again today. Please fix this immediately. This is not acceptable.

Your software needs to answer four questions:

1. What is the customer's issue?
2. How urgent is it?
3. Should the request go to a human?
4. What should the software do next?

There are two broad ways to approach this.

### Approach A: Ask a conventional LLM

You send the customer's message to an LLM such as GPT or Claude and ask it to analyze the request.

It might respond:

```json
{
  "category": "billing",
  "urgency": "high",
  "needs_human": true,
  "reason": "The customer was charged after canceling."
}
```

This is useful. Modern LLMs can also be instructed to return structured JSON.

But your application still needs to ensure that the response conforms to the expected schema, contains valid values, and provides useful decisions.

Structured output solves an important formatting problem. It doesn't, by itself, guarantee that the decisions are correct or that any confidence estimates are reliable.

### Approach B: Ask Jev

TypeSafe designed Jev specifically for decision-oriented tasks.

You provide the customer's message and ask defined questions about it.

![Approach B, ask Jev. The customer message plus any state you hold goes into one call carrying three typed questions, a Choice for department, a Score for urgency and a Noul for needs_human. Three typed answers come back: department billing at confidence 1.00, urgency 2.99 of 3 at confidence 0.99, and needs_human at probability 0.80. Your own plain Python then compares those numbers against thresholds and decides the queue. Approach A returned the string high and the boolean true, neither of which can be compared against a threshold.](images/ask_jev_flow.png)

Jev is designed to return typed decisions with associated probabilities and confidence information, rather than generate a free-form response. Its underlying model architecture and training approach, according to its documentation, are specifically optimized for this purpose.

The key difference is the intended interface and its output: a conversational LLM generates language; Jev is designed to supply judgments or decisions that software downstream can use directly.

That doesn't mean Jev is automatically more accurate for every task. You need to evaluate both approaches on your own workload.However, TypeSafe contends that it's most suited for workloads or workflows that are heavily decision based.

## 1.2 The most important concept: Separate judgment from action

Consider this business rule:

> If a customer has a billing problem and the request is urgent, prioritize the ticket. If the situation is uncertain, send it to a human.

This contains two fundamentally different kinds of work.

**Work A: Understand the situation.**

* Is this a billing problem?
* Is it urgent?
* Does it need human attention?

Human language is messy. These questions benefit from AI judgment.

**Work B: Apply the business policy.**

* If the category is billing and urgency is high, prioritize the ticket.
* If human review is required, send the ticket to the review queue.
* Otherwise, follow the normal processing path.

These are rules that ordinary software can implement reliably.

Jev is designed to handle the first kind of work. Your program handles the second.

Here's the architecture:

![Separate judgment from action: a customer message goes to Jev, which returns a department label with confidence, an urgency level with confidence, and a needs-human probability; those three values feed the plain Python if-statements in decide(), which sends the ticket to triage when the department is unclear, to a human queue when the needs-human probability is high, and to the automated queue otherwise, with a priority flag set by urgency alone](images/judgment_vs_action.png)

This separation is important because you can change the business policy without changing the questions you ask Jev.

For example, today you might escalate urgent billing issues. Tomorrow, you might also escalate every case involving a suspected duplicate charge.

You can change the rules in your application rather than asking the AI to invent and follow a new policy on every request.

TypeSafe describes this approach as building decision-based workflows from narrow AI judgments and explicit program logic.

## 1.3 Why not just use an LLM with JSON output?

This is the most important technical question in the tutorial.

Suppose you ask a conventional LLM to return:

```json
{
  "category": "billing",
  "urgency": 8,
  "needs_human": true
}
```

There are three distinct requirements.

| Requirement | What it means | Does JSON alone solve it? |
|---|---|---|
| **Structure** | The response has the expected fields and data types. | It helps, especially with schema-constrained output. |
| **Correctness** | The category and urgency actually fit the situation. | No. |
| **Reliable uncertainty** | The system can use meaningful uncertainty estimates when deciding whether to act. | No. |

Jev is designed to address all three, with a particular emphasis on structured decisions and well-behaved probabilities.

A probability is **calibrated** when it means what it says. Out of 100 answers that come back at 80%, about 80 should be right.

Calibration does not mean every individual prediction is correct, and it can change when the workload or data changes.

That's why the application still needs testing, monitoring, and appropriate safeguards.

### When should you choose Jev?

Jev is worth evaluating when:

* You need to make many narrow decisions.
* The possible output types can be defined in advance.
* You want probabilities to inform downstream logic.
* Speed and cost matter because the model sits inside a larger workflow.
* You want the surrounding code to control what actions are permitted.

A conventional LLM is generally a better fit when the main task is to write an explanation, produce a report, generate code, or engage in open-ended reasoning.

And you can combine the two: Jev can classify or prioritize a request, while an LLM writes the response. You do exactly that in Lesson 6.

## Lesson 1 exercise: You are the architect

Answer this one before you look at any code.

You're designing the customer-service application. The customer says:

> "I canceled my subscription last week, but you charged my credit card again today. Please fix this immediately."

Your system needs to make three decisions:

**A. Category:** Which department should handle this? Billing · Shipping · Technical support · Sales

**B. Urgency:** How urgent is the request, on a scale from 0 to 10?

**C. Escalation:** Should the application send this request to a human? Yes · No

Here's your challenge:

**Which of these decisions should be handled by AI judgment, and which should be handled by ordinary program logic?**

Choose the best answer:

* **Option 1:** Let Jev make all three decisions and execute the resulting actions.
* **Option 2:** Let Jev classify the request and assess urgency; let ordinary code apply the escalation policy.
* **Option 3:** Use ordinary code for everything, including understanding the customer's message.

Think about it before reading further.

<details>
<summary><strong>Reveal the answer and explanation</strong></summary>

**Option 2** is the best starting architecture.

Jev can assess the message and return structured judgments about the category and urgency. It can also assess whether human review is warranted.

Your code then applies the policy. For example:

```python
if urgency >= 8 or needs_human_review:
    send_to_human_queue()
else:
    process_normally()
```

This is illustrative pseudocode, not a Jev API implementation.

The important distinction is that **the AI interprets the situation, while the application controls the resulting action.**

For a real system, the threshold should be selected using representative evaluation data. High-impact actions should also have appropriate safeguards.

**One thing to unlearn already.** The "scale from 0 to 10" above is how people naturally pose urgency, and it is not how you will ask Jev. Lesson 2.3 replaces it with three or four levels you describe in words, because "8 out of 10" makes the model invent a scale while "needs an answer today, losing money or access" is something it can actually recognize.

Why not the others?

* **Option 1** puts the model in charge of actions. Now a wrong judgment is a wrong *action*, and your policy lives somewhere you cannot unit-test.
* **Option 3** is where most teams start and get stuck. No amount of `if "refund" in message` reliably distinguishes "I want a refund" from "I already got a refund, but you charged me again."

</details>

## What you've learned

You now have the foundational mental model:

1. Jev is designed for **decisions**, not text generation.
2. AI judgment and program logic are **different responsibilities**.
3. Typed output makes AI decisions easier to integrate into software.
4. Probabilities can help applications handle uncertainty, provided they're properly evaluated.
5. Jev and conventional LLMs can complement each other.

**Docs for this lesson:** [Introduction](https://docs.typesafe.ai/introduction) · [System One](https://docs.typesafe.ai/concepts/system-one) · [Example use cases](https://docs.typesafe.ai/concepts/use-case-map)

## Your next lesson

Next you get hands-on with Jev's three core primitives:

* **Noul:** Is this statement true, or does this condition hold?
* **Choice:** Which of these options best fits?
* **Score:** How well does this item fit a criterion?

You'll apply each one to the customer-service example, then learn how to turn a business requirement into a question Jev can answer.

---

# Lesson 2: The three primitives

Lesson 1 ended with four questions about a customer message. Jev doesn't accept free-form questions. It accepts *typed* ones. There are exactly three types, and learning to pick between them is most of the skill.

| Primitive | The question it answers | What comes back |
|---|---|---|
| **`Noul`** | Is this statement true, or does this condition hold? | one probability, `0.0`–`1.0` |
| **`Choice`** | Which of these options best fits? | the winning label, plus a probability for each |
| **`Score`** | How well does this item fit a criterion? | a number on your rubric, plus a probability for each level |

Three types is a feature, not a limitation. Because the shape of the answer is fixed before the call, your code can branch on it without parsing anything.

## 2.1 `Noul`: does this proposition hold?

A `Noul` takes a *statement*, not a question, and returns the probability that the statement is true. The name is TypeSafe's, and it is easiest to read as "one proposition": you assert something, and Jev tells you how likely it is to hold.

```python
Noul(instructions="The customer is asking for a refund")
```

The answer has one field:

```python
answer.noul   # 0.93  probability the statement is true
```

Note what's missing: there's no `confidence`. For a `Noul` the probability *is* the uncertainty. `0.93` means "almost certainly true", `0.5` means "I genuinely don't know", `0.04` means "almost certainly false". A second number would be redundant.

**Writing good `Noul` instructions.** Phrase the statement so that "true" is unambiguous:

| Instead of | Write |
|---|---|
| "Is this urgent?" | "The message conveys urgency or time-sensitivity" |
| "Refund?" | "The customer is explicitly requesting a refund" |
| "Is this OK to automate?" | "This ticket can be resolved without a human agent" |

A statement that bundles two claims gives you one number for two facts. "The customer is angry and wants a refund" is two facts, so ask two `Noul`s instead.

## 2.2 `Choice`: which option fits?

A `Choice` is classification. You name the options and describe each one; Jev distributes probability across them.

```python
Choice(
    instructions="Which team should handle this",
    criteria={
        "billing":   "Payment or subscription issues",
        "technical": "Bugs or integration problems",
        "sales":     "Pricing or account questions",
    },
)
```

The answer carries three fields:

```python
answer.choice         # "billing"  the option with the highest probability
answer.confidence     # 0.95       how concentrated the probabilities are
answer.probabilities  # {"billing": 0.96, "technical": 0.02, "sales": 0.02}
```

**The descriptions are the work.** `criteria` can be a plain list of labels, but a dict lets you say where the boundaries are. Boundaries are where classification goes wrong. "Payment or subscription issues" tells Jev that a cancellation complaint belongs in billing, not sales.

**The options must cover the input.** Jev always returns one of your options, even for a message that fits none of them. If your tickets include things outside your taxonomy, add an explicit `"other"` option rather than letting them land in whichever category is least wrong.

## 2.3 `Score`: how well does this fit?

A `Score` is a rubric. You describe each level in order, lowest to highest, and Jev places the item on it.

```python
Score(
    instructions="How frustrated the customer appears",
    criteria=[
        "Calm, just stating facts",      # level 0
        "Frustrated but civil",          # level 1
        "Very angry, strong language",   # level 2
    ],
)
```

The answer:

```python
answer.score          # 1.4  probability-weighted average of the levels
answer.confidence     # 0.78
answer.probabilities  # {0: 0.08, 1: 0.44, 2: 0.48}
answer.legend         # {0: "Calm, just stating facts", 1: ..., 2: ...}
```

**`score` is continuous even though the rubric is discrete.** `1.4` doesn't mean "level 1.4 of the rubric". It means the probability mass sits between levels 1 and 2. That's useful: you get a smooth quantity to threshold or sort on, out of a rubric a human can read.

**Describe the levels, don't number them.** "Needs an answer today, losing money or access" is a level Jev can recognize. "8 out of 10" is not. It pushes the model to invent a scale, which is the vagueness you were trying to remove.

**Keep the rubric short.** Three to five levels that a human reviewer could apply consistently. If *you* can't tell level 6 from level 7, neither can the model, and the extra resolution is noise.

## 2.4 Translating a business requirement into questions

This is the part that takes practice. Go back to Lesson 1's rule:

> If a customer has a billing problem and the request is urgent, prioritize the ticket. If the situation is uncertain, send it to a human.

Read it again and separate the two kinds of work. Every **noun that describes the situation** becomes a question. Every **verb that changes the world** stays in your code.

| Phrase in the rule | Kind of work | Where it goes |
|---|---|---|
| "has a billing problem" | judgment, one of several categories | `Choice` |
| "the request is urgent" | judgment, a matter of degree | `Score` |
| "send it to a human" | judgment about whether a person is needed | `Noul` |
| "prioritize the ticket" | **action** | your code |
| "if ... and ..." | **policy** | your code |
| "the situation is uncertain" | **threshold on a probability** | your code |

That last row is the one people miss. "Uncertain" is not something to ask Jev. You don't add a `Noul(instructions="I am unsure")`. Uncertainty already comes back with every answer, as `confidence` or as a `noul` near `0.5`. Your code decides what counts as too uncertain.

So the rule becomes three questions:

```python
QUESTIONS = {
    "department":  Choice(instructions="Which team should handle this ticket", criteria=DEPARTMENTS),
    "urgency":     Score(instructions="How soon this customer needs an answer", criteria=URGENCY),
    "needs_human": Noul(instructions="This ticket needs a human agent rather than an automated reply"),
}
```

and one policy, in Python, that reads their answers. You write both for real in Lesson 4.

## Lesson 2 exercise: Pick the primitive

For each requirement, which primitive would you reach for? One is a trick.

1. "Flag reviews that contain personal information."
2. "Sort incoming leads so sales calls the most promising first."
3. "Send the ticket to whichever of our six support queues fits."
4. "Reject comments that violate our policy, but have a human look at the borderline ones."
5. "Write an apology email to the customer."

<details>
<summary><strong>Reveal the answers</strong></summary>

1. **`Noul`.** `Noul(instructions="This review contains personally identifying information")` is a single proposition, and you threshold the probability.

2. **`Score`.** You need an *ordering*, and `score` is a continuous number you can sort on. A `Choice` between "hot"/"warm"/"cold" throws away the ranking inside each bucket.

3. **`Choice`.** Six mutually exclusive labels. Describe each queue's boundary in `criteria`, and consider a seventh `"other"` option for tickets that fit none.

4. **`Noul`**, and then *two* thresholds in your code, not two questions:

   ```python
   p = answers["violates_policy"].noul
   if p > 0.90:   reject()
   elif p > 0.60: send_to_human_review()
   else:          allow()
   ```

   The middle band is a policy choice, not a model output. This is the pattern Lesson 5 is about.

5. **None of them.** This is text generation, so give it to a conventional LLM. The useful combination is Jev deciding *whether* and *how urgently* to apologize, and an LLM writing the words. Lesson 6 builds exactly that.

</details>

## What you've learned

1. Jev asks exactly three kinds of question: `Noul`, `Choice`, `Score`.
2. `Noul` returns one probability and needs no separate confidence, because the probability *is* the uncertainty.
3. `Choice` and `Score` return a value plus `confidence` plus the full `probabilities` distribution.
4. The `criteria` descriptions are where the real design happens. You are drawing boundaries, not naming labels.
5. Nouns describing the situation become questions; verbs that change the world stay in your code. "Uncertain" is a threshold, never a question.

**Docs for this lesson:** [Primitives overview](https://docs.typesafe.ai/primitives) · [`Noul`](https://docs.typesafe.ai/primitives/noul) · [`Choice`](https://docs.typesafe.ai/primitives/choice) · [`Score`](https://docs.typesafe.ai/primitives/score)

## Your next lesson

Enough reading. Next you make a real API call: construct a client, send the customer message with one question attached, and read every field that comes back.

---

# Lesson 3: Your first API call

Every Jev call has the same three parts:

```
state  +  questions   ->   client.system_one()   ->   typed answers
```

* **`state`**: the thing you want judged. Here, the customer's message.
* **`questions`**: a dict mapping names you choose to `Noul` / `Choice` / `Score` objects.
* **the response**: `.answers`, a dict with the same keys you asked under.

Start with one question.

In [19]:
response = client.system_one(
    state=TICKET,
    questions={
        'is_urgent': Noul(instructions='The message conveys urgency or time-sensitivity'),
    },
)
answer = response.answers['is_urgent']

print(f'p(urgent) = {answer.noul:.3f}')


p(urgent) = 0.980


That's the whole API. A statement in, a probability out.

Now use it the way an application would, as a condition rather than a sentence:

In [ ]:
if answer.noul > 0.75:
    print('-> fast lane')
else:
    print('-> normal queue')

The `0.75` is yours. Jev reported a probability; your code decided what to do about it. That's the Lesson 1 split in two lines of real code.

## 3.1 `state` doesn't have to be a string

`state` accepts any JSON-serializable value, and in a real application it rarely is just the message.

There's no `State` object to construct. Unlike the questions, `state` is a plain value you hand straight to the call. The diagram at the top of this lesson puts `state` and `questions` side by side, but only the second one is built out of primitives.

Give Jev the context a human agent would have:

In [20]:
ticket_state = {
    'message': TICKET,
    'customer': {'plan': 'pro', 'tenure_months': 14, 'lifetime_value_usd': 1680},
    'subscription': {'status': 'canceled', 'canceled_on': '2026-10-01'},
    'recent_charges': [{'date': '2026-10-08', 'amount_usd': 49.00}],
}

answer = client.system_one(
    state=ticket_state,
    questions={
        'billing_error': Noul(
            instructions='The customer was charged after their subscription was canceled',
        ),
    },
).answers['billing_error']

print(f'p(billing error) = {answer.noul:.3f}')

p(billing error) = 0.970


Notice the question got sharper because the state got richer. With only the message, "was the customer actually charged after canceling?" is the customer's claim. With the subscription record and the charge list in `state`, it's a question about data Jev can actually check.

**Rule of thumb:** put in `state` whatever a competent human would need to answer the question, and nothing more. Extra irrelevant fields cost tokens and can distract the model.

## 3.2 What the response object carries

`.answers` is the part you use, but the response has more: which model answered, and what it cost.

In [21]:
response = client.system_one(
    state=TICKET,
    questions={'is_urgent': Noul(instructions='The message conveys urgency or time-sensitivity')},
)

print('model     :', response.model)
print('request_id:', response.request_id)
print('tokens in :', response.usage.input_tokens)
print('tokens out:', response.usage.output_tokens)
print('answers   :', list(response.answers))

model     : jev-1.14.0
request_id: req_01a184b6436e78e1bd7cc9eea2c1f51c
tokens in : 303
tokens out: 23
answers   : ['is_urgent']


`request_id` is worth logging. When an answer looks wrong, that's the identifier you quote. The gateway returns it as the `x-typesafe-request-id` response header rather than a field in the body; the client lifts it onto the response object for you.

## 3.3 Configuring the client

`JevClient()` with no arguments reads the environment:

| Variable | What it sets |
|---|---|
| `DATABRICKS_HOST` | your workspace; the gateway URL is derived from it |
| `DATABRICKS_TOKEN` | the token the gateway authenticates with |
| `JEV_MODEL` | the Unity Catalog endpoint path to call |
| `JEV_GATEWAY_URL` | optional: the full endpoint URL, if you don't want it derived from the host |

Explicit arguments win over environment variables, and a missing variable raises immediately with its name rather than failing later as a `401`:

```python
client = JevClient(model='catalog.schema.jev_endpoint', timeout=10.0)
```

You can also override the model for a single call, which is how you A/B two registered endpoints on the same question:

```python
client.system_one(state=TICKET, questions=QUESTIONS, model='catalog.schema.other_endpoint')
```

Unlike the hosted API, the gateway serves **one registered endpoint per model path** rather than a catalogue you can list. The model you get is whatever Unity Catalog is pointed at. So to see what you're actually talking to, read it off the client and the response:

In [22]:
print('UC endpoint   :', client.model)
print('gateway URL   :', client.url)
print('model version :', response.model)   # e.g. jev-1.14.0

UC endpoint   : jules_catalog.uaigw_jevl.uaigw_jev1_endpoint
gateway URL   : https://e2-dogfood.staging.cloud.databricks.com/ai-gateway/typesafe/v1/systemone
model version : jev-1.14.0


## Lesson 3 exercise: Sharpen a vague question

Someone on your team writes this:

```python
client.system_one(
    state='Customer seems unhappy about the charge',
    questions={'bad': Noul(instructions='Is this bad?')},
)
```

Three things are wrong with it. Find them, then fix the call.

<details>
<summary><strong>Reveal the answers</strong></summary>

**1. The instruction is a question, not a statement.** `Noul` returns the probability that a *proposition* holds. "Is this bad?" leaves it unclear what `0.9` would even mean. Bad for whom, in what way? Rewrite as an assertion you could check: `"The customer was charged in error"`.

**2. "Bad" bundles several facts.** Unhappy? Charged wrongly? Likely to churn? Those are different questions with different thresholds. One number can't serve all three.

**3. The `state` is already a judgment.** `"Customer seems unhappy about the charge"` is somebody's summary. The original message is gone, and so is the subscription record that would settle whether the charge was an error. You've asked Jev to judge a paraphrase.

A better version:

```python
client.system_one(
    state={
        'message': TICKET,                                    # the raw text
        'subscription': {'status': 'canceled', 'canceled_on': '2026-10-01'},
        'recent_charges': [{'date': '2026-10-08', 'amount_usd': 49.00}],
    },
    questions={
        'charged_in_error': Noul(
            instructions='The customer was charged after their subscription was canceled',
        ),
        'wants_refund': Noul(
            instructions='The customer is requesting a refund of the disputed charge',
        ),
    },
)
```

Raw state, two separate propositions, each phrased so that "true" is unambiguous.

</details>

## What you've learned

1. One method does everything: `client.system_one(state=..., questions=...)`.
2. `state` is any JSON-serializable value, so pass the context a human agent would need.
3. `.answers` keys match your question keys, so the response shape is known before the call.
4. The response also carries `model`, `usage`, and `request_id`. Log the last one.
5. The client reads `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, and `JEV_MODEL`; explicit arguments take precedence.

**Docs for this lesson:** [Quickstart](https://docs.typesafe.ai/introduction/quickstart) · [State](https://docs.typesafe.ai/concepts/state) · [Python SDK usage](https://docs.typesafe.ai/sdk/python/usage) · [Models](https://docs.typesafe.ai/models)

## Your next lesson

Your customer message needed three judgments, and so far you've asked for one at a time. Next you ask all three in a single call. It is faster, and it turns out to be the better design.

---

# Lesson 4: Asking several questions at once

Lesson 1 listed four things the software needed to know about one customer message. Three of them are questions for Jev. The fourth one, *what should the software do next*, was never a question at all. It is the policy, and you write it in 4.3 at the end of this lesson.

So: three questions. The naive implementation is three calls. The right implementation is one.

`questions` is a dict, so put them all in it:

In [23]:
DEPARTMENTS = {
    'billing': 'Charges, invoices, refunds, duplicate payments',
    'shipping': 'Delivery, tracking, lost or late packages',
    'technical': 'Errors, crashes, logins, broken integrations',
    'account': 'Passwords, email changes, closing an account',
}

URGENCY = [
    'No rush, a general question',
    'Normal, wants an answer this week',
    'Needs an answer soon, is blocked',
    'Needs an answer today, losing money or access',
]

QUESTIONS = {
    'department': Choice(
        instructions='Which team should handle this ticket',
        criteria=DEPARTMENTS,
    ),
    'urgency': Score(
        instructions='How soon this customer needs an answer',
        criteria=URGENCY,
    ),
    'needs_human': Noul(
        instructions='This ticket needs a human agent rather than an automated reply',
    ),
}

response = client.system_one(state=TICKET, questions=QUESTIONS)
answers = response.answers

print(f"department  : {answers['department'].choice}")
print(f"urgency     : {answers['urgency'].score:.2f} of {len(URGENCY) - 1}")
print(f"needs_human : p = {answers['needs_human'].noul:.2f}")

department  : billing
urgency     : 2.99 of 3
needs_human : p = 0.84


Three different primitives, three different answer types, one HTTP round trip, instead of three.

## 4.1 Why one call beats three

**Cost and latency.** The `state` is sent once instead of three times. For a ticket with customer history attached, the state is most of the payload, so three calls cost roughly three times as much and take three times as long.

**Consistency.** All answers come from one pass over the same state. Three separate calls can disagree with each other: one call decides the ticket is `shipping`, another decides it's not urgent *because* it read the situation differently. One call gives you a coherent reading.

**It matches how your code consumes it.** Your policy function needs all three values before it can decide anything. Fetching them separately just means three chances to fail before you can act.

## 4.2 Reading the answers back

Each answer type exposes its own fields, and the response also groups them by type for convenience. Every answer also carries `.type`, one of `'noul'`, `'choice'` or `'score'`. That is how you tell them apart when you loop over answers you didn't write out by hand.

**One asymmetry to know about before it bites you.** `response.nouls` gives you bare floats, because a `Noul` answer has nothing else in it. `response.choices` and `response.scores` give you the answer *objects*, because those carry `confidence` and `probabilities` too. So `response.nouls['needs_human']` is a number you can compare directly, while `response.choices['department']` needs a `.choice` on the end. When in doubt, use `response.answers`, where every value is an object:

In [24]:
# Careful: these three are not symmetric.
print('nouls  :', response.nouls)                  # name -> float
print('choices:', response.choices['department'])   # name -> answer object
print('scores :', response.scores['urgency'])       # name -> answer object


nouls  : {'needs_human': 0.84}
choices: ChoiceAnswer(choice='billing', confidence=1.0, probabilities={'billing': 1.0, 'shipping': 0.0, 'technical': 0.0, 'account': 0.0}, type='choice')
scores : ScoreAnswer(score=2.99, confidence=0.99, legend={0: 'No rush, a general question', 1: 'Normal, wants an answer this week', 2: 'Needs an answer soon, is blocked', 3: 'Needs an answer today, losing money or access'}, probabilities={0: 0.0, 1: 0.0, 2: 0.01, 3: 0.99}, type='score')


In [25]:
# The full distributions, not just the winners.
dept = answers['department']
print(f'department = {dept.choice}  (confidence {dept.confidence:.2f})')
for label, p in sorted(dept.probabilities.items(), key=lambda kv: -kv[1]):
    print(f'  {label:<12} {p:6.1%}  {DEPARTMENTS[label]}')

urg = answers['urgency']
print(f'\nurgency = {urg.score:.2f}  (confidence {urg.confidence:.2f})')
for level, p in sorted(urg.probabilities.items()):
    print(f'  {level}  {p:6.1%}  {urg.legend[level]}')

department = billing  (confidence 1.00)
  billing      100.0%  Charges, invoices, refunds, duplicate payments
  shipping       0.0%  Delivery, tracking, lost or late packages
  technical      0.0%  Errors, crashes, logins, broken integrations
  account        0.0%  Passwords, email changes, closing an account

urgency = 2.99  (confidence 0.99)
  0    0.0%  No rush, a general question
  1    0.0%  Normal, wants an answer this week
  2    1.0%  Needs an answer soon, is blocked
  3   99.0%  Needs an answer today, losing money or access


The distributions are the interesting part, and most code throws them away. `department.choice` tells you the winner; `department.probabilities` tells you whether it was a landslide or a coin flip between `billing` and `account`. Lesson 5 is about using that.

## 4.3 The policy, finally

Now write Work B from Lesson 1. Ordinary Python, with no model anywhere in it:

In [26]:
CONFIDENT_ENOUGH = 0.60      # below this, we do not trust the department label
HUMAN_ABOVE = 0.75           # p(needs a human) above this goes to a person
PRIORITY_AT_OR_ABOVE = 2.5   # urgency at or above this jumps the queue


def decide(answers):
    """Turn Jev's answers into an action. Pure Python, trivially testable."""
    department = answers['department']
    urgency = answers['urgency']
    needs_human = answers['needs_human']

    priority = urgency.score >= PRIORITY_AT_OR_ABOVE

    if department.confidence < CONFIDENT_ENOUGH:
        return 'triage', priority, f'department unclear ({department.confidence:.2f})'
    if needs_human.noul > HUMAN_ABOVE:
        return f'{department.choice}-human', priority, f'needs a human (p={needs_human.noul:.2f})'
    return f'{department.choice}-auto', priority, f'confident ({department.confidence:.2f})'


queue, priority, why = decide(answers)
print(f"{queue}{' · PRIORITY' if priority else ''}  ({why})")

billing-human · PRIORITY  (needs a human (p=0.84))


Read `decide()` and notice what it is: three named constants, one comparison each, and no model anywhere in it. A security reviewer, a support manager, or a unit test can all read it. Change `PRIORITY_AT_OR_ABOVE` to `1.5` and the business policy changes, without touching a single question.

That's the payoff Lesson 1 promised, in working code.

In [27]:
# Because decide() takes answers rather than a ticket, you can test the policy
# with no API key and no network at all.
from types import SimpleNamespace

fake = {
    'department': SimpleNamespace(choice='billing', confidence=0.95),
    'urgency': SimpleNamespace(score=2.9),
    'needs_human': SimpleNamespace(noul=0.91),
}
print(decide(fake))

fake['department'] = SimpleNamespace(choice='billing', confidence=0.41)  # ambiguous
print(decide(fake))

('billing-human', True, 'needs a human (p=0.91)')
('triage', True, 'department unclear (0.41)')


## Lesson 4 exercise: Add a requirement

Product adds a rule:

> Customers on the enterprise plan always get a human, regardless of how routine the ticket looks.

Where does that change go: a new question, a new `criteria` entry, or new code? Write it.

<details>
<summary><strong>Reveal the answer</strong></summary>

**New code.** Nothing about the plan tier requires judgment. It is a field you already have.

```python
def decide(answers, customer):
    ...
    if customer['plan'] == 'enterprise':
        return f'{department.choice}-human', priority, 'enterprise plan'
    ...
```

Two things to notice.

**Don't ask Jev what you already know.** A `Noul(instructions="This customer is on the enterprise plan")` would hand you a *probability* for a fact that is already sitting in your database as a string. You'd be adding uncertainty to certainty, and paying tokens for it.

**Where you put the check matters.** Placed before the confidence gate, enterprise tickets skip triage even when the department is unclear, so they reach a human, but possibly the wrong team's human. Placed after, an ambiguous enterprise ticket goes to triage first. Both are defensible; the point is that it's a *policy* question you can see and argue about, because it's four lines of Python rather than a sentence in a prompt.

</details>

## What you've learned

1. `questions` is a dict, so ask everything you need in one `system_one` call.
2. One call is cheaper, faster, and internally consistent, because the state is read once.
3. `response.nouls` gives bare floats; `.choices` and `.scores` give answer objects. `response.answers` is uniformly objects, which is why the rest of the tutorial uses it.
4. A policy function that takes *answers* instead of a *ticket* is testable with no network and no key.
5. New business rules usually mean new code, not new questions. Don't ask Jev what your database already knows.

**Docs for this lesson:** [How to build with System One](https://docs.typesafe.ai/concepts/how-to-build-with-system-one) · [Parallel questions cookbook](https://docs.typesafe.ai/cookbooks/parallel_questions) · [Fan-out pattern](https://docs.typesafe.ai/patterns/fan-out)

## Your next lesson

You've been using thresholds like `0.75` and `0.60` without justifying them. Next: what `confidence` actually measures, which primitives report it, and how to choose the boundaries for your own stakes.

---

# Lesson 5: Working with uncertainty

Every answer so far came with an uncertainty signal you mostly ignored. That signal is the reason to use Jev at all. It is what lets your software know when *not* to act.

The cells in this lesson read the `answers` from Lesson 4, so run that call first.

## 5.1 What `confidence` measures

`confidence` is **how concentrated the probability is**, not how likely the answer is to be right.

> `confidence` is `1` when all the probability sits on one outcome, and `0` when it's spread evenly across all of them.

With four departments, `0.25` on each is a pure guess, so confidence is `0.0`. All of it on `billing` is `1.0`. Real answers land in between, and the number tells you how close to a guess this one was.

A `Score` works the same way, with one twist: the rubric is *ordered*, so how far apart the candidate levels are matters. Torn between levels 2 and 3 counts as more confident than torn between 0 and 3, because adjacent levels are nearly the same answer.

A `Noul` has **no** `confidence` field, because the probability already carries the uncertainty: a value near `0.5` is the model saying it's unsure. If you need a number comparable to a `Choice`'s confidence, compute it yourself:

```python
certainty = abs(2 * answer.noul - 1)   # 0.5 -> 0.0,  0.0 or 1.0 -> 1.0
```


In [28]:
# confidence summarizes the distribution, so look at the two side by side.
dept = answers['department']

for label, p in sorted(dept.probabilities.items(), key=lambda kv: -kv[1]):
    print(f'  {label:<12} {p:.2f}')
print(f'  {"-" * 18}')
print(f'  {"confidence":<12} {dept.confidence:.2f}  <- how concentrated the above is')

p = answers['needs_human'].noul
print(f'\n  {"noul":<12} {p:.2f}')
print(f'  {"|2p - 1|":<12} {abs(2 * p - 1):.2f}  <- a Choice-comparable certainty')


  billing      1.00
  shipping     0.00
  technical    0.00
  account      0.00
  ------------------
  confidence   1.00  <- how concentrated the above is

  noul         0.84
  |2p - 1|     0.68  <- a Choice-comparable certainty


## 5.2 The three bands

The pattern TypeSafe recommends, and the one worth defaulting to:

| Band | What your software does |
|---|---|
| **High confidence** | Act automatically. |
| **Medium confidence** | Proceed with caution. Ask the user to confirm, flag for review, or gather more information. |
| **Low confidence** | Do not act. Route to a human, request clarification, or fall back to a different system. |

**Where you draw those boundaries depends on the stakes.** A `0.5` floor may be fine for choosing which help article to show. A destructive, irreversible action deserves something above `0.9`, and arguably a human regardless.

Start with conservative thresholds, test them with your own data, and adjust as you observe results.

The router you build in Lesson 6 uses a simpler shape than three bands: two gates in order, department confidence first and then `needs_human`. Three bands is the general pattern to reach for. Two gates is what this particular problem needed, and matching the pattern to the problem is your call rather than the model's. There is no principled way to pick `0.85` from first principles. Until you have measured it against labelled inputs, any threshold is a number you picked because it looked round, and the exercise below works through what it takes to defend one.

## Lesson 5 exercise: Set the thresholds

You're routing moderation decisions with:

```python
Noul(instructions='This comment violates our community policy')
```

Three actions are available: **delete the comment**, **send it to a human moderator**, **leave it up**. Deleting is visible to the author and hard to undo; leaving a violation up has reputational cost; human review costs about $0.40 per comment and you get 50,000 comments a day.

Write the thresholds, and say what you'd need to know to defend them.

<details>
<summary><strong>Reveal one reasonable answer</strong></summary>

```python
p = answers['violates_policy'].noul

if p > 0.97:    delete()                # very high bar: the action is visible and hard to undo
elif p > 0.55:  human_review()          # wide band, because mistakes here are expensive both ways
else:           leave_up()
```

**Why the bands are shaped that way.** The two errors are not symmetric. Deleting a legitimate comment is a visible insult to a real person; leaving a borderline one up for an hour is usually recoverable. Asymmetric costs mean asymmetric thresholds: `0.97` to delete, but only `0.55` to ask a human.

**What makes it affordable.** The review band is `0.55–0.97`, and its width sets your bill. If 8% of comments land in it, that's 4,000 reviews a day and $1,600. If 30% land in it, it's $6,000 and you need to narrow the band or accept more automated deletions.

**What you'd need to defend it:**

* **The distribution of `p` over real traffic.** The thresholds alone tell you nothing about cost; only the histogram does. You cannot budget for a band whose width in volume you haven't measured.
* **Accuracy inside each band**, from human-labelled samples. "Of the comments we auto-delete above `0.97`, how many were actually violations?" If that's 99.8%, the bar might be too high and you're paying for reviews you don't need. If it's 94%, it's too low.
* **The cost of each error type**, in whatever currency your organization actually argues in: support tickets, churn, regulatory exposure.
* **Drift monitoring.** Calibration can change when the workload changes. A new product launch, a brigading campaign, or a different language mix all shift the distribution. Alert on the band widths, not just on accuracy.

Any specific numbers in this answer are a starting point for *your* measurement, not a recommendation.

</details>

## What you've learned

1. `confidence` measures how concentrated the probability is. `1` means all the mass on one outcome, `0` means evenly spread.
2. `Choice` and `Score` both report a `confidence`; a `Score` counts distant disagreement as less confident than adjacent; `Noul` has none, because its probability already is the uncertainty.
3. Act, review, escalate is the pattern uncertainty turns into behaviour through. Lesson 6's router uses two gates instead, because that is what the problem needed.
4. High confidence is not correctness. Low confidence is sometimes the right answer about a genuinely ambiguous input.
5. Thresholds come from labelled data and the relative cost of each error, and need re-checking as the workload drifts.

**Docs for this lesson:** [Confidence](https://docs.typesafe.ai/confidence) · [Confidence routing](https://docs.typesafe.ai/patterns/confidence-routing) · [Classification using confidence](https://docs.typesafe.ai/cookbooks/classification_using_confidence)

## Your next lesson

Last lesson: assemble everything into a working application, run it over a batch of tickets, and hand the text-writing to a conventional LLM. Jev decides, the LLM speaks.

---

# Lesson 6: Building a complete AI workflow

Everything you wrote in Lesson 4 now lives in [`router.py`](router.py). Same questions, same thresholds, same `if` statements, lifted out of notebook cells into a module you can import, test, and deploy. Open it alongside this lesson; nothing in it is new.

![Jev returns decisions, not words: the same customer message sent to a conversational LLM comes back as prose your code must parse, while sent to Jev it comes back as a typed label, a rubric level, and a probability that ordinary program logic can branch on directly](images/jev_decisions_not_words.png)

### Two halves, two functions

| In the file | What it is |
|---|---|
| `DEPARTMENTS`, `URGENCY`, `QUESTIONS` | **judgment**: what you ask Jev. Nothing here mentions queues or priorities. |
| `CONFIDENT_ENOUGH`, `HUMAN_ABOVE`, `PRIORITY_AT_OR_ABOVE`, `decide()` | **policy**: what your software does about the answers. |

The comments in the file name those two regions, and the functions follow the same line:

```python
decide(answers)  ->  Routing             # the policy: ifs on numbers. No network, no key.
route(ticket)    ->  (Routing, answers)  # ask Jev, then call decide()
```

`decide()` is the one to read. It is your Lesson 4 policy function, with the three values it returned wrapped in a `Routing` instead of a bare tuple. It still takes *answers* rather than a ticket, so you can call it with numbers you invented, which is what made it testable without a model in the loop. `route()` is the thin wrapper: one Jev call, then hand off.

`Routing` is a dataclass of three fields with a `__str__`, which is why `print(routing)` below reads as a sentence rather than a tuple:

```python
routing, answers = route(TICKET, client=client)

routing.queue      # str   'triage', 'billing-auto', 'billing-human', ...
routing.priority   # bool  does it jump the queue
routing.why        # str   the reason, e.g. 'needs a human, p=0.91'
```

`route()` hands back `answers` as well as the decision, because a decision with no evidence attached can't be logged or debugged. 6.2 uses those same numbers to brief an LLM.

### Why the tutorial needs the file at all

- **6.1 routes six tickets.** Without a module, every cell that wants a decision has to re-paste the questions and the policy.
- **The judgment/policy split stops being advice and becomes structure.** Someone who needs to know what this application is permitted to do reads the policy half and nothing else. That's the Lesson 1.2 architecture, enforced by where the code sits in the file.
- **It's the deployable artifact.** A notebook cell is something you ran once; `router.py` imports without Jupyter, so a test suite, a web handler, or a nightly batch job can call `route()`.

The gateway call underneath it is in [`jev_client.py`](jev_client.py): one POST, three question types, typed answers back. That's the whole of what this tutorial needed from the API.


In [29]:
from jevai.router import route

routing, answers = route(TICKET, client=client)
print(routing)

billing-human · PRIORITY  (needs a human, p=0.83)


## 6.1 Running a batch

A router is only interesting across a range of inputs. Here are six tickets covering the easy cases, the urgent ones, and two that are deliberately ambiguous:

In [30]:
TICKETS = [
    TICKET,
    'Where is my order? It was supposed to arrive Tuesday and the tracking page is blank.',
    'How do I change the email address on my account?',
    "The API has been returning 500s for 20 minutes. Our checkout is down and we're losing sales.",
    'Thanks for the help yesterday, much appreciated!',
    "I can't log in and I also think I've been double-billed. Not sure which is the real problem.",
]

for i, ticket in enumerate(TICKETS, 1):
    routing, answers = route(ticket, client=client)
    print(f'{i}. {ticket[:66]}{"..." if len(ticket) > 66 else ""}')
    print(f'   -> {routing}')
    print(f'      department={answers["department"].choice} '
          f'conf={answers["department"].confidence:.2f}  '
          f'urgency={answers["urgency"].score:.2f}  '
          f'p(human)={answers["needs_human"].noul:.2f}\n')

1. I canceled my subscription last week, but you charged my credit ca...
   -> billing-human · PRIORITY  (needs a human, p=0.84)
      department=billing conf=1.00  urgency=2.98  p(human)=0.84

2. Where is my order? It was supposed to arrive Tuesday and the track...
   -> shipping-auto  (confident (1.00) and automatable)
      department=shipping conf=1.00  urgency=1.82  p(human)=0.64

3. How do I change the email address on my account?
   -> account-auto  (confident (1.00) and automatable)
      department=account conf=1.00  urgency=0.21  p(human)=0.27

4. The API has been returning 500s for 20 minutes. Our checkout is do...
   -> technical-human · PRIORITY  (needs a human, p=0.95)
      department=technical conf=1.00  urgency=2.98  p(human)=0.95

5. Thanks for the help yesterday, much appreciated!
   -> technical-auto  (confident (0.61) and automatable)
      department=technical conf=0.61  urgency=0.01  p(human)=0.18

6. I can't log in and I also think I've been double-billed. Not s

Look at the last two.

The thank-you note fits none of the four departments, so the probabilities spread out and confidence falls to right about `0.60`, which is exactly where `CONFIDENT_ENOUGH` sits. Run the cell a few times and watch it flip: `0.56` routes to `triage`, `0.61` routes to `technical-auto`. Same ticket, same code, different queue.

That flapping is worth more than a clean answer would be. Two lessons fall out of it. First, the "add an explicit `other` option" point from Lesson 2, in real output: this ticket has nowhere correct to go, and no threshold can fix a taxonomy with a missing category. Second, a hard cutoff on a continuous signal is unstable for anything that lands near it. When you pick a threshold, check how many of your real tickets cluster around it, not just how it scores the obvious cases. An input at `0.90` is a decision; an input at `0.60` is a coin flip you chose to take.

The login-plus-billing ticket is genuinely two problems. Low confidence here isn't the model failing; it's the model telling you your taxonomy doesn't fit this input. Routing it to a human is the right answer, and the confidence gate got you there without a special case.

## 6.2 Jev decides, an LLM writes

Lesson 1 said you can combine the two. This is what that looks like: Jev produces the structured judgment, your policy decides what happens, and a conversational LLM writes the customer-facing text *within* the decision already made. Prose is what an LLM is good at. Authority is not.

The chat model comes through the same Unity AI Gateway that serves Jev. The gateway has two routes, and that is the only difference: Jev answers typed questions on `/ai-gateway/typesafe/v1/systemone`, while chat models answer on `/ai-gateway/mlflow/v1`. The second route speaks an OpenAI-compatible API, which is why the cell below imports `openai`. That package is only the client library, and nothing in this notebook calls OpenAI.

`JEVAI_MODEL` is a Unity Catalog endpoint path, just like `JEV_MODEL`, so it is an indirection rather than a model name. Ask the response what actually answered and you get something like `global.anthropic.claude-opus-4-8`, which is the point: the policy above never depended on which model sits behind that path.

In [31]:
import os

from openai import OpenAI

# Same Unity AI Gateway that serves Jev, on its other route. Jev answers typed
# questions on /ai-gateway/typesafe/v1/systemone; chat models answer on
# /ai-gateway/mlflow/v1, which speaks an OpenAI-compatible API.
CHAT_URL = os.environ.get(
    'JEVAI_GATEWAY_URL',
    f"{os.environ['DATABRICKS_HOST'].rstrip('/')}/ai-gateway/mlflow/v1",
)
# A Unity Catalog endpoint path, exactly like JEV_MODEL. From jevai/.env, and a
# missing one raises KeyError naming itself rather than failing as a 404.
LLM_MODEL = os.environ['JEVAI_MODEL']

llm = OpenAI(api_key=os.environ['DATABRICKS_TOKEN'], base_url=CHAT_URL)


def reply(ticket, routing, answers):
    """Jev and the policy have already decided. The LLM only writes."""
    brief = (
        f'Department: {answers["department"].choice}\n'
        f'Urgency: {answers["urgency"].legend[round(answers["urgency"].score)]}\n'
        f'Going to: {routing.queue}{" (priority)" if routing.priority else ""}'
    )
    completion = llm.chat.completions.create(
        model=LLM_MODEL,
        # Generous, because reasoning models spend part of this budget thinking
        # before they emit any prose. At 250 the reply comes back truncated.
        max_tokens=1000,
        messages=[
            {'role': 'system', 'content': (
                'You write short, warm customer-support acknowledgements. Two or three '
                'sentences. Tell the customer what will happen next, based only on the '
                'routing brief. Promise nothing the brief does not say. No refund amounts, '
                'no dates, no apologies for things the brief does not mention.'
            )},
            {'role': 'user', 'content': f'Ticket:\n{ticket}\n\nRouting brief:\n{brief}'},
        ],
    )
    # Reasoning models on Databricks return content as a list of blocks
    # ('reasoning' then 'text') rather than a plain string. Take the text.
    content = completion.choices[0].message.content
    if isinstance(content, list):
        content = '\n'.join(b['text'] for b in content if b.get('type') == 'text')
    return content.strip()


routing, answers = route(TICKET, client=client)
print('gateway :', CHAT_URL)
print('endpoint:', LLM_MODEL)
print(routing, '\n')
print(reply(TICKET, routing, answers))

gateway : https://e2-dogfood.staging.cloud.databricks.com/ai-gateway/mlflow/v1
endpoint: jules_catalog.uaigw_claude.uaigw-claude-endpoint
billing-human · PRIORITY  (needs a human, p=0.85) 

Thanks for reaching out about this charge. I've flagged your ticket as a priority and sent it straight to our billing team, who will review it today. You'll hear back from them with the details soon.


Note where the authority sits. The LLM never saw the thresholds and never chose a queue. It was handed a decision and asked for words. If it hallucinates a refund amount, that's a prose bug in a message, not a payment. The routing was settled before it was invoked.

This is the division of labour worth taking away: **Jev for judgments your code branches on, an LLM for language your users read.**

## 6.3 What to build next

Three projects that use what you now know, in increasing order of difficulty:

| Project | The questions | The policy |
|---|---|---|
| **Support router** | you just built it | thresholds on confidence and urgency |
| **RAG reranker** | one `Score` per retrieved chunk: "how well does this passage answer the question?" | sort by `score`, keep the top *k*, drop everything below a floor |
| **Agent gatekeeper** | `Noul`s per proposed action: "the user explicitly asked for this", "this destroys unrecoverable data" | three bands per action; irreversible ones need a human |

The reranker is the best second project. It's one `Score` applied *n* times in parallel, the output is a sort key, and you can measure the improvement against your existing retrieval with data you already have. See the [reranking cookbook](https://docs.typesafe.ai/cookbooks/rerank_typesafe).

The gatekeeper is the one that matters most in production, and it's where this tutorial hands off.

## Lesson 6 exercise: Find the authority

Here are two implementations of the same feature. One of them has a defect you should be able to name now.

**Version A**

```python
answers = client.system_one(state=ticket, questions={
    'action': Choice(instructions='What should we do with this ticket', criteria={
        'refund': 'Issue a full refund immediately',
        'escalate': 'Send to a human agent',
        'reply': 'Send an automated reply',
    }),
}).answers
ACTIONS[answers['action'].choice](ticket)
```

**Version B**

```python
answers = client.system_one(state=ticket, questions={
    'charged_in_error': Noul(instructions='The customer was charged after canceling'),
    'wants_refund': Noul(instructions='The customer is requesting a refund'),
}).answers

if answers['charged_in_error'].noul > 0.9 and answers['wants_refund'].noul > 0.8:
    if ticket_amount_usd <= AUTO_REFUND_LIMIT:
        issue_refund(ticket_amount_usd)
    else:
        escalate('refund above limit')
else:
    escalate('unclear')
```

<details>
<summary><strong>Reveal the answer</strong></summary>

**Version A lets the model choose the action.** Three defects follow from that one mistake:

1. **No dollar limit exists anywhere.** `refund` is a label, so there's no amount to check and nowhere to check it. A `$12` refund and a `$12,000` refund take the same code path.
2. **Confidence is unused.** `.choice` is read without ever looking at `.confidence`, so a 34%-vs-33%-vs-33% three-way tie issues a refund just as readily as a 99% certainty.
3. **The policy is inside `criteria`.** "Issue a full refund immediately" is a *business rule* written in a string that gets sent to a model. To change the rule you edit a prompt, and you cannot unit-test it.

**Version B asks about the situation and decides in code.** `charged_in_error` and `wants_refund` are facts about the world; the limit, the thresholds, and the escalation are policy. The policy branch is testable with no network, and a reviewer can read every number that matters.

This is the Lesson 1 exercise again, Option 1 versus Option 2, with real code on it. It's also the single most common mistake in production Jev applications: putting the verb in `criteria`.

Version B is still improvable. The two `Noul` thresholds are independent, so a ticket at `0.89`/`0.95` escalates while one at `0.91`/`0.81` refunds, which may not be the trade-off you want. Neither number came from labelled data yet.

</details>

## What you've learned

1. A complete Jev application is a questions block, a policy function, and nothing else.
2. Ambiguous inputs produce low confidence, and a confidence gate handles them without special cases.
3. Jev and conversational LLMs compose cleanly: Jev decides, the LLM writes, and the LLM never holds authority.
4. `Score`-per-item is a reranker; `Noul`-per-action is a gatekeeper. Same two regions, different questions.
5. Whenever you can't find the policy in the code, it's in a prompt. That's the bug.

**Docs for this lesson:** [Intent routing](https://docs.typesafe.ai/patterns/intent-routing) · [Composite scoring](https://docs.typesafe.ai/patterns/composite-scoring) · [LLM guardrails cookbook](https://docs.typesafe.ai/cookbooks/llm_guardrails) · [All cookbooks](https://docs.typesafe.ai/cookbooks)

---

# Where to go from here

You can now explain Jev to someone without mentioning the API, which was the point:

> Software asks typed questions about its state. Jev answers them with values and probabilities. Ordinary code reads those answers and decides what happens. The model supplies evidence; your program owns the policy.

## Reference

* [Jev documentation](https://docs.typesafe.ai/): start at [Introduction](https://docs.typesafe.ai/introduction)
* [Use case map](https://docs.typesafe.ai/concepts/use-case-map): which pattern fits your problem
* [Patterns](https://docs.typesafe.ai/patterns) and [cookbooks](https://docs.typesafe.ai/cookbooks): working code for fan-out, reranking, guardrails, function calling
* [Python SDK reference](https://docs.typesafe.ai/sdk/python/api): every type and field, and what `jev_client.py` mirrors
* [Machine learning primer](https://docs.typesafe.ai/introduction/machine-learning-primer): if the probability discussion in Lesson 5 went too fast
* [Databricks AI Gateway](https://docs.databricks.com/aws/en/ai-gateway/): the gateway serving Jev here, including rate limits and usage tracking